# Part 1 Notebook - MQA + MCQ (Datathon 2026)

This notebook is designed for the **first scoring block**:
1. `MQA`: data quality and integrity checks
2. `MCQ`: reproducible answers with exact numbers

Outputs are exported to `../eda_results/scoring/`.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from IPython.display import display

DATA_DIR = Path('../dataset')
OUT_DIR = Path('../eda_results/scoring')
OUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.4f}'.format)

sales = pd.read_csv(DATA_DIR / 'sales.csv', parse_dates=['Date'])
orders = pd.read_csv(DATA_DIR / 'orders.csv', parse_dates=['order_date'])
order_items = pd.read_csv(DATA_DIR / 'order_items.csv', low_memory=False)
payments = pd.read_csv(DATA_DIR / 'payments.csv')
shipments = pd.read_csv(DATA_DIR / 'shipments.csv', parse_dates=['ship_date', 'delivery_date'])
returns = pd.read_csv(DATA_DIR / 'returns.csv', parse_dates=['return_date'])
reviews = pd.read_csv(DATA_DIR / 'reviews.csv', parse_dates=['review_date'])
customers = pd.read_csv(DATA_DIR / 'customers.csv', parse_dates=['signup_date'])
products = pd.read_csv(DATA_DIR / 'products.csv')
promotions = pd.read_csv(DATA_DIR / 'promotions.csv', parse_dates=['start_date', 'end_date'])
geography = pd.read_csv(DATA_DIR / 'geography.csv')
inventory = pd.read_csv(DATA_DIR / 'inventory.csv', parse_dates=['snapshot_date'])
web = pd.read_csv(DATA_DIR / 'web_traffic.csv', parse_dates=['date'])

datasets = {
    'sales': sales,
    'orders': orders,
    'order_items': order_items,
    'payments': payments,
    'shipments': shipments,
    'returns': returns,
    'reviews': reviews,
    'customers': customers,
    'products': products,
    'promotions': promotions,
    'geography': geography,
    'inventory': inventory,
    'web_traffic': web,
}
print('Loaded datasets:', ', '.join(datasets.keys()))

Loaded datasets: sales, orders, order_items, payments, shipments, returns, reviews, customers, products, promotions, geography, inventory, web_traffic


## MQA - Table-level Quality

In [2]:
rows = []
for name, df in datasets.items():
    total_cells = len(df) * len(df.columns)
    missing_pct = float(df.isna().sum().sum() / total_cells * 100) if total_cells else 0.0
    rows.append({
        'dataset': name,
        'rows': len(df),
        'cols': len(df.columns),
        'missing_pct_total': missing_pct,
    })

mqa_overview = pd.DataFrame(rows).sort_values('rows', ascending=False)
display(mqa_overview)

mqa_overview.to_csv(OUT_DIR / 'mqa_overview.csv', index=False)
print('Saved:', OUT_DIR / 'mqa_overview.csv')

print('\nTop missing columns by dataset:')
for name, df in datasets.items():
    miss = (df.isna().mean() * 100).sort_values(ascending=False)
    miss = miss[miss > 0]
    if not miss.empty:
        print(f'- {name}:', {k: round(v, 2) for k, v in miss.head(4).to_dict().items()})

,dataset,rows,cols,missing_pct_total
2,order_items,714669,7,23.0440
3,payments,646945,4,0.0000
1,orders,646945,8,0.0000
4,shipments,566067,4,0.0000
7,customers,121930,7,0.0000
6,reviews,113551,7,0.0000
11,inventory,60247,17,0.0000
10,geography,39948,4,0.0000
5,returns,39939,7,0.0000
0,sales,3833,3,0.0000


Saved: ../eda_results/scoring/mqa_overview.csv

Top missing columns by dataset:
- order_items: {'promo_id_2': 99.97, 'promo_id': 61.34}
- promotions: {'applicable_category': 80.0}


## MQA - Integrity and Logic Checks

In [3]:
order_ids = set(orders['order_id'])
product_ids = set(products['product_id'])
promo_ids = set(promotions['promo_id'])

integrity = pd.DataFrame([
    {'check': 'Duplicate order_id in orders', 'value': int(orders['order_id'].duplicated().sum())},
    {'check': 'Duplicate order_id in payments', 'value': int(payments['order_id'].duplicated().sum())},
    {'check': 'Duplicate product_id in products', 'value': int(products['product_id'].duplicated().sum())},
    {'check': 'Orphan order_id in order_items', 'value': int((~order_items['order_id'].isin(order_ids)).sum())},
    {'check': 'Orphan order_id in payments', 'value': int((~payments['order_id'].isin(order_ids)).sum())},
    {'check': 'Orphan order_id in shipments', 'value': int((~shipments['order_id'].isin(order_ids)).sum())},
    {'check': 'Orphan order_id in returns', 'value': int((~returns['order_id'].isin(order_ids)).sum())},
    {'check': 'Orphan order_id in reviews', 'value': int((~reviews['order_id'].isin(order_ids)).sum())},
    {'check': 'Orphan product_id in order_items', 'value': int((~order_items['product_id'].isin(product_ids)).sum())},
    {'check': 'Orphan product_id in returns', 'value': int((~returns['product_id'].isin(product_ids)).sum())},
    {'check': 'Unknown promo_id in order_items', 'value': int((~order_items['promo_id'].dropna().isin(promo_ids)).sum())},
    {'check': 'Delivery before ship date', 'value': int((shipments['delivery_date'] < shipments['ship_date']).sum())},
    {'check': 'Negative/zero payment value', 'value': int((payments['payment_value'] <= 0).sum())},
    {'check': 'Negative discount amount', 'value': int((order_items['discount_amount'] < 0).sum())},
])

display(integrity)
integrity.to_csv(OUT_DIR / 'mqa_integrity_checks.csv', index=False)
print('Saved:', OUT_DIR / 'mqa_integrity_checks.csv')

sales_missing_days = pd.date_range(sales['Date'].min(), sales['Date'].max(), freq='D').difference(sales['Date'])
web_missing_days = pd.date_range(web['date'].min(), web['date'].max(), freq='D').difference(web['date'])

print('\nDate continuity:')
print('- sales missing daily dates:', len(sales_missing_days))
print('- web missing daily dates:', len(web_missing_days))

constant_inventory_cols = [c for c in inventory.columns if inventory[c].nunique(dropna=False) == 1]
print('- inventory constant columns:', constant_inventory_cols)

,check,value
0,Duplicate order_id in orders,0
1,Duplicate order_id in payments,0
2,Duplicate product_id in products,0
3,Orphan order_id in order_items,0
4,Orphan order_id in payments,0
5,Orphan order_id in shipments,0
6,Orphan order_id in returns,0
7,Orphan order_id in reviews,0
8,Orphan product_id in order_items,0
9,Orphan product_id in returns,0


Saved: ../eda_results/scoring/mqa_integrity_checks.csv

Date continuity:
- sales missing daily dates: 0
- web missing daily dates: 0
- inventory constant columns: ['reorder_flag']


## MCQ - Reproducible Answers (Q1-Q10)

In [4]:
# Q1
q1_df = orders.loc[orders['order_status'] == 'delivered', ['customer_id', 'order_date']].drop_duplicates()
q1_df = q1_df.sort_values(['customer_id', 'order_date'])
q1_df['gap_days'] = q1_df.groupby('customer_id')['order_date'].diff().dt.days
q1_value = float(q1_df['gap_days'].dropna().median())

# Q2
products['gross_margin'] = (products['price'] - products['cogs']) / products['price']
q2_tbl = products.groupby('segment')['gross_margin'].mean().sort_values(ascending=False)
q2_value = q2_tbl.index[0]

# Q3
q3_tbl = returns.merge(products[['product_id', 'category']], on='product_id', how='left')
q3_tbl = q3_tbl.loc[q3_tbl['category'] == 'Streetwear', 'return_reason'].value_counts()
q3_value = q3_tbl.index[0]

# Q4
q4_tbl = web.groupby('traffic_source')['bounce_rate'].mean().sort_values()
q4_value = q4_tbl.index[0]

# Q5
q5_value = float(order_items['promo_id'].notna().mean() * 100)

# Q6
order_counts = orders.groupby('customer_id').size().rename('total_orders').reset_index()
q6_tbl = customers.loc[customers['age_group'].notna(), ['customer_id', 'age_group']].merge(order_counts, on='customer_id', how='left')
q6_tbl['total_orders'] = q6_tbl['total_orders'].fillna(0)
q6_tbl = q6_tbl.groupby('age_group')['total_orders'].mean().sort_values(ascending=False)
q6_value = q6_tbl.index[0]

# Q7
gross_line = order_items.assign(revenue=lambda d: d['quantity'] * d['unit_price'])
q7_tbl = (
    gross_line[['order_id', 'revenue']]
    .merge(orders[['order_id', 'zip']], on='order_id', how='left')
    .merge(geography[['zip', 'region']], on='zip', how='left')
    .groupby('region')['revenue']
    .sum()
    .sort_values(ascending=False)
)
q7_value = q7_tbl.index[0]

# Q8
q8_tbl = orders.loc[orders['order_status'] == 'cancelled', 'payment_method'].value_counts()
q8_value = q8_tbl.index[0]

# Q9
ret_size = returns.merge(products[['product_id', 'size']], on='product_id', how='left').groupby('size').size()
ord_size = order_items.merge(products[['product_id', 'size']], on='product_id', how='left').groupby('size').size()
q9_tbl = (ret_size / ord_size).sort_values(ascending=False)
q9_value = q9_tbl.index[0]

# Q10
q10_tbl = payments.groupby('installments')['payment_value'].mean().sort_values(ascending=False)
q10_value = int(q10_tbl.index[0])

mcq_answers = pd.DataFrame([
    {'qid': 'Q1', 'answer': q1_value, 'note': 'Median inter-order gap (days) among delivered orders'},
    {'qid': 'Q2', 'answer': q2_value, 'note': 'Highest average gross-margin segment'},
    {'qid': 'Q3', 'answer': q3_value, 'note': 'Most common return reason in Streetwear'},
    {'qid': 'Q4', 'answer': q4_value, 'note': 'Lowest average bounce-rate traffic source'},
    {'qid': 'Q5', 'answer': round(q5_value, 2), 'note': 'Percent of order_items rows with promo_id not null'},
    {'qid': 'Q6', 'answer': q6_value, 'note': 'Age group with highest avg orders per customer'},
    {'qid': 'Q7', 'answer': q7_value, 'note': 'Region with highest total revenue'},
    {'qid': 'Q8', 'answer': q8_value, 'note': 'Most used payment method among cancelled orders'},
    {'qid': 'Q9', 'answer': q9_value, 'note': 'Size with highest return-rate proxy'},
    {'qid': 'Q10', 'answer': q10_value, 'note': 'Installment plan with highest avg payment value'},
])

display(mcq_answers)
mcq_answers.to_csv(OUT_DIR / 'mcq_answers.csv', index=False)
print('Saved:', OUT_DIR / 'mcq_answers.csv')

,qid,answer,note
0,Q1,178.0000,Median inter-order gap (days) among delivered ...
1,Q2,Standard,Highest average gross-margin segment
2,Q3,wrong_size,Most common return reason in Streetwear
3,Q4,email_campaign,Lowest average bounce-rate traffic source
4,Q5,38.6600,Percent of order_items rows with promo_id not ...
5,Q6,55+,Age group with highest avg orders per customer
6,Q7,East,Region with highest total revenue
7,Q8,credit_card,Most used payment method among cancelled orders
8,Q9,S,Size with highest return-rate proxy
9,Q10,6,Installment plan with highest avg payment value


Saved: ../eda_results/scoring/mcq_answers.csv


## Export MQA + MCQ Snapshot

In [5]:
snapshot = {
    'datasets': {name: {'rows': len(df), 'cols': len(df.columns)} for name, df in datasets.items()},
    'mcq_answers': mcq_answers.set_index('qid')['answer'].to_dict(),
}
with open(OUT_DIR / 'part1_snapshot.json', 'w', encoding='utf-8') as f:
    json.dump(snapshot, f, indent=2, default=str)
print('Saved:', OUT_DIR / 'part1_snapshot.json')

Saved: ../eda_results/scoring/part1_snapshot.json
